# Hamiltonian Monte Carlo & NUTS — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Use gradients and momentum to propose distant moves with high acceptance
HMC simulates Hamiltonian dynamics to move through continuous posteriors. These examples compute leapfrog steps, energy error, acceptance, trajectory length, and the NUTS U-turn condition.

In [ ]:
# 1) one leapfrog step for standard normal target
def U(q): return 0.5*q*q
def gradU(q): return q
q,p=1.0,0.3; eps=0.2; p_half=p-0.5*eps*gradU(q); q_new=q+eps*p_half; p_new=p_half-0.5*eps*gradU(q_new)
plt.figure(figsize=(6,3)); plt.arrow(q,0,q_new-q,0,head_width=.05,length_includes_head=True); plt.xlim(0.8,1.1); plt.title('leapfrog move in position')
assert abs(q_new-1.04)<1e-12

In [ ]:
# 2) Hamiltonian energy is nearly conserved
H0=U(q)+0.5*p*p; H1=U(q_new)+0.5*p_new*p_new
plt.figure(figsize=(6,3)); plt.bar(['before','after'],[H0,H1]); plt.title('small energy error')
assert abs(H1-H0)<0.01

In [ ]:
# 3) MH correction uses energy difference
accept=min(1,math.exp(H0-H1)); plt.figure(figsize=(6,3)); plt.bar(['accept prob'],[accept]); plt.ylim(0,1.05); plt.title(f'accept={accept:.3f}')
assert accept>0.99

In [ ]:
# 4) multiple leapfrog steps make a distant proposal
qs=[q]; ps=[p]
qq,pp=q,p
for _ in range(12):
    ph=pp-0.5*eps*gradU(qq); qq=qq+eps*ph; pp=ph-0.5*eps*gradU(qq); qs.append(qq); ps.append(pp)
plt.figure(figsize=(6,3)); plt.plot(qs,ps,'-o',ms=3); plt.xlabel('q'); plt.ylabel('p'); plt.title('Hamiltonian trajectory')
assert max(qs)-min(qs)>0.3

In [ ]:
# 5) NUTS stops when trajectory turns back toward the start
start=qs[0]; dots=[(qq-start)*pp for qq,pp in zip(qs,ps)]; plt.figure(figsize=(6,3)); plt.plot(dots,'-o'); plt.axhline(0,c='k'); plt.title('U-turn when displacement · momentum < 0')
assert min(dots)<0